# Data loading

In [ ]:
import copy
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statistics import mean, stdev
from scipy.stats import norm, multivariate_normal

In [ ]:
rcupd = {
    'figure.figsize': (5, 4),
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': 'cm',
    'font.size': 12,
    'axes.formatter.limits': (-3, 3),
}
plt.rcParams.update(rcupd)

In [ ]:
data_files = [
    '2025-11-04/RERTR5_V6018G.csv',
    '2025-11-04/RERTR12_L1P755.csv',
]

# Convenience functions

In [ ]:
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

def mod_metrics(mod, X_test, y_test):
    y_pred = mod.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    print(
        ' R2: ', r2, '\n',
        'RMSE: ', rmse, '\n',
        'MAE: ', mae
    )

In [ ]:
def pred_vs_actual(mod, X_test, y_test, tt):
    y_pred = mod.predict(X_test)

    plt.figure(figsize=(5,4))
    plt.rcParams.update({'font.size': 16})

    plt.scatter(y_test, y_pred, s=15)

    minv = int(min(min(y_test), min(y_pred)))
    maxv = int(max(max(y_test), max(y_pred)))
    val = list(range(minv, maxv))
    
    plt.plot(val, val, color='k', ls='--', label='y=x')

    plt.title(tt)
    plt.xlabel(r'Test data (swelling \%)')
    plt.ylabel(r'Surrogate pred. (swelling \%)')
    plt.legend()
    plt.show()

# Preprocessing

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [ ]:
def load_data(fileName):
    jar = pd.read_csv(fileName)
    # vResol issue
    jar['vResol'] = jar['vResol'] * 1e18

    col_names = jar.columns[1:-2]

    X = jar.iloc[:, 1:-2].to_numpy()
    y = jar.iloc[:, -2].to_numpy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.7, random_state=13
    )

    return X_train, X_test, y_train, y_test, col_names

In [ ]:
X_train_lo, X_test_lo, y_train_lo, y_test_lo, col_names = load_data(data_files[0])
X_train_hi, X_test_hi, y_train_hi, y_test_hi, _ = load_data(data_files[1])

In [ ]:
X_comb = np.concatenate((X_train_lo, X_train_hi), axis=0)

In [ ]:
xscaler = MinMaxScaler()
xscaler.fit(X_comb)

In [ ]:
X_train_lo = xscaler.transform(X_train_lo)
X_test_lo = xscaler.transform(X_test_lo)

X_train_hi = xscaler.transform(X_train_hi)
X_test_hi = xscaler.transform(X_test_hi)

# GP

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, DotProduct, RBF

In [ ]:
kern = ConstantKernel(constant_value_bounds=(1e-5,1e7)) * Matern()
reg_lo = GaussianProcessRegressor(
    kernel=kern,
    alpha=1e-2,
    n_restarts_optimizer=9,
    random_state=42
).fit(X_train_lo, y_train_lo)
print(reg_lo.kernel_)

mod_metrics(reg_lo, X_test_lo, y_test_lo)
pred_vs_actual(reg_lo, X_test_lo, y_test_lo, r'GP low $F_d$')

In [ ]:
kern = ConstantKernel(constant_value_bounds=(1e-5,1e7)) * Matern()
reg_hi = GaussianProcessRegressor(
    kernel=kern,
    alpha=1e-3,
    n_restarts_optimizer=9,
    random_state=42
).fit(X_train_hi, y_train_hi)
print(reg_hi.kernel_)

mod_metrics(reg_hi, X_test_hi, y_test_hi)
pred_vs_actual(reg_hi, X_test_hi, y_test_hi, r'GP high $F_d$')

# Sensitivity analysis

In [ ]:
from SALib.analyze.sobol import analyze
from SALib.sample.sobol import sample

In [ ]:
def get_sobol(mod, st):
    problem = {
        'num_vars': 9,
        'names': col_names,
        'bounds': [[0, 1]] * 9
    }

    param_vals = sample(problem, 1024, calc_second_order=False)
    Y = mod.predict(param_vals)

    Si = analyze(problem, Y,
                 calc_second_order=False, print_to_console=True)

    fig, axes = plt.subplots(1, 2, figsize=(8, 5))

    axes[0].bar(col_names, Si['S1'],
                yerr=Si['S1_conf'], capsize=3,
                color='teal', label=r'S$_i$')
    axes[0].tick_params(axis='x', rotation=80)
    axes[0].legend()
    axes[0].set_ylim([0, 1])

    axes[1].bar(col_names, Si['ST'],
                yerr=Si['ST_conf'], capsize=3,
                color='teal', label=r'S$_{T_{i}}$')
    axes[1].tick_params(axis='x', rotation=80)
    axes[1].set_ylim([0, 1])
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    #plt.savefig(f'sobol_{st}.pdf')

In [ ]:
get_sobol(reg_lo, 'lo')

In [ ]:
get_sobol(reg_hi, 'hi')

# Multivariate gaussian

In [ ]:
def cr_multivar_gaussian(mu1, sig1, mu2, sig2, cov12):
    means = np.array([mu1, mu2])
    cov_matrix = np.array([[sig1**2, cov12],
                            [cov12, sig2**2]])
    mvn = multivariate_normal(mean=means, cov=cov_matrix)
    return mvn

In [ ]:
mvn = cr_multivar_gaussian(10.7, 2.64, 32, 2.64, 0)

In [ ]:
x, y = np.mgrid[0.7:20.7:0.1, 22:42:0.1]
pos = np.dstack((x, y))

fig = plt.figure(figsize=(5,4))
ax = fig.add_subplot(111)
cs = ax.contourf(x, y, mvn.pdf(pos), cmap='Purples', levels=10)
plt.colorbar(cs, label='PDF')

ax.set_xlabel(r"Observed swelling (\%) at low $F_d$")
ax.set_ylabel(r"Observed swelling (\%) at high $F_d$")
plt.tight_layout()
plt.show()
#plt.savefig('mvn_expt.pdf')

# MCMC sampler

In [ ]:
def proposal_dist(X, sig):
    ret = []
    
    for el in X:
        prop = np.random.normal(el, sig)
        ret.append(prop)

    assert len(X) == len(ret)
    return ret

In [ ]:
def mcmc_sampler(num_param, initial_state, proposal_sig,
                 surrogates, num_samples):
    samples = [initial_state]
    accepted = 0

    for ii in range(num_samples):
        current_state = samples[-1]
        proposed_state = proposal_dist(current_state, proposal_sig)

        valid = True
        for xx in proposed_state:
            if xx < 0 or xx > 1:
                valid = False
                break

        fs1_curr_mean, fs1_curr_std = surrogates[0].predict([current_state], return_std=True)
        fs2_curr_mean, fs2_curr_std = surrogates[1].predict([current_state], return_std=True)
        
        fs1_prop_mean, fs1_prop_std = surrogates[0].predict([proposed_state], return_std=True)
        fs2_prop_mean, fs2_prop_std = surrogates[1].predict([proposed_state], return_std=True)

        curr_mvn = cr_multivar_gaussian(
            10.7,
            (2.64**2 + fs1_curr_std[0]**2)**0.5,
            32,
            (2.64**2 + fs2_curr_std[0]**2)**0.5,
            0
        )

        prop_mvn = cr_multivar_gaussian(
            10.7,
            (2.64**2 + fs1_prop_std[0]**2)**0.5,
            32,
            (2.64**2 + fs2_prop_std[0]**2)**0.5,
            0
        )

        acceptance_ratio = (
            prop_mvn.pdf([fs1_prop_mean[0], fs2_prop_mean[0]])
            / curr_mvn.pdf([fs1_curr_mean[0], fs2_curr_mean[0]])
        )

        if valid and np.random.rand() < acceptance_ratio:
            current_state = proposed_state
            accepted += 1

        samples.append(current_state)

    print(f"Acceptance rate: {accepted / num_samples}")
    return np.array(samples)

In [ ]:
need_to_sample = False

In [ ]:
if need_to_sample:
    hey1 = mcmc_sampler(
        9,
        np.random.rand(9),
        0.11,
        [reg_lo, reg_hi],
        200000
    )
    np.save('hey1_gp.npy', hey1)
else:
    hey1 = np.load('hey1_gp.npy')

In [ ]:
if need_to_sample:
    hey2 = mcmc_sampler(
        9,
        np.random.rand(9),
        0.11,
        [reg_lo, reg_hi],
        200000
    )
    np.save('hey2_gp.npy', hey2)
else:
    hey2 = np.load('hey2_gp.npy')

# Trace/Hist

In [ ]:
old1 = xscaler.inverse_transform(hey1)
# fix vResol
old1[:, -2] = old1[:, -2] * 1e-18

old2 = xscaler.inverse_transform(hey2)
# fix vResol
old2[:, -2] = old2[:, -2] * 1e-18

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(10, 10))

for i, ax in enumerate(axes.flatten()):
    cdat1 = old1[:,i]
    cavg1 = np.cumsum(cdat1) / np.arange(1, len(cdat1)+1)
    ax.plot(cdat1, lw=0.1, alpha=0.7, zorder=1, rasterized=True)
    ax.plot(cavg1, c='k', zorder=2)
    
    cdat2 = old2[:,i]
    cavg2 = np.cumsum(cdat2) / np.arange(1, len(cdat2)+1)
    ax.plot(cdat2, ls='--', lw=0.1, alpha=0.7, zorder=1, rasterized=True)
    ax.plot(cavg2, c='r', zorder=2)
    
    ax.set_xlabel(col_names[i])
    #ax.set_ylim([0, 1])

#fig.delaxes(axes[1,3])
fig.supylabel('Parameter values')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(10, 10))

for i, ax in enumerate(axes.flatten()):
    sns.histplot(old1[:,i], ax=ax, stat='density', kde=True)
    sns.histplot(old2[:,i], ax=ax, stat='density', kde=True)
    ax.set_xlabel(col_names[i])
    ax.set_ylabel('')
    #ax.set_xlim([0, 1])

#fig.delaxes(axes[1,3])
fig.supylabel('Density')
plt.tight_layout()
plt.show()

# Posterior correlation

In [ ]:
cold = np.concatenate((old1[::100], old2[::100]))

len(cold)

oldd = pd.DataFrame(cold, columns=col_names)

In [ ]:
# plt.figure(figsize=(10, 10))
# sns.pairplot(
#     pd.DataFrame(oldd),
#     diag_kind='hist',
#     diag_kws=dict(kde=True),
#     plot_kws=dict(levels=6, bw_adjust=2.0, cmap='viridis'),
#     kind='kde',
#     corner=True
# )
# 
# plt.show()

In [ ]:
# plt.figure(figsize=(10, 10))
# sns.heatmap(oldd.corr(), annot=True, cmap='coolwarm')
# 
# plt.show()

In [ ]:
oldd.describe()

In [ ]:
for i in range(9):
    print(np.mean(oldd.iloc[:, i]),
          np.std(oldd.iloc[:, i]))

In [ ]:
for i in range(9):
    print(np.percentile(oldd.iloc[:, i], 2.5),
          np.percentile(oldd.iloc[:, i], 97.5))

# FUQ

In [ ]:
chey = np.concatenate((hey1[::100], hey2[::100]))

len(chey)

In [ ]:
res = []
for i in range(4000):
    res_lo = reg_lo.predict([chey[-i]])[0]
    res_hi = reg_hi.predict([chey[-i]])[0]
    res.append([res_lo, res_hi])

swel_lo = np.array(res)[:, 0]
swel_hi = np.array(res)[:, 1]

#plt.hist2d(x, y)
sns.kdeplot(x=swel_lo, y=swel_hi,
            fill=True, cmap='Purples',
            cbar=True, cbar_kws={'label': 'Density'})
#sns.jointplot(x=x, y=y, kind='kde', fill=True, cmap='Purples')

plt.xlabel(r"Swelling (\%) at low $F_d$")
plt.ylabel(r"Swelling (\%) at high $F_d$")

plt.xlim([0.7, 20.7])
plt.ylim([22, 42])

plt.tight_layout()
plt.show()
#plt.savefig('fuq_mvn.pdf')

In [ ]:
mus = [10.7, 32]
sigs = [2.64, 2.64]

for swel, mu, sig in zip([swel_lo, swel_hi], mus, sigs):
    x = np.linspace(mu - 4*sig, mu + 4*sig, 100)
    y = norm(mu, sig).pdf(x)
    plt.plot(x, y, 'r', label='Obs. with noise')
    plt.fill_between(x, y, color='r', alpha=0.5)

    sns.histplot(swel, binwidth=0.5,
                 ec='k', stat='density', label='Forward propagation')

    plt.xlim([mu - 5*sig, mu + 5*sig])

    plt.xlabel(r'Fuel Swelling (\%)')
    plt.legend(fontsize=10)

    plt.tight_layout()
    plt.show()
    #plt.savefig(f'fuq_{mu}.pdf')
    #plt.close()

# FUQ (reduced parameters)

In [ ]:
post_means = np.mean(chey, axis=0)

In [ ]:
res = []
for i in range(4000):
    newarr = np.concatenate((chey[-i][:4], post_means[4:]))
    res_lo = reg_lo.predict([newarr])[0]
    res_hi = reg_hi.predict([newarr])[0]
    res.append([res_lo, res_hi])

nom_lo = np.array(res)[:, 0]
nom_hi = np.array(res)[:, 1]

#plt.hist2d(x, y)
sns.kdeplot(x=nom_lo, y=nom_hi,
            fill=True, cmap='Purples',
            cbar=True, cbar_kws={'label': 'Density'})
#sns.jointplot(x=x, y=y, kind='kde', fill=True, cmap='Purples')

plt.xlabel(r"Swelling (\%) at low $F_d$")
plt.ylabel(r"Swelling (\%) at high $F_d$")

plt.xlim([0.7, 20.7])
plt.ylim([22, 42])

plt.tight_layout()
plt.show()

In [ ]:
mus = [10.7, 32]
sigs = [2.64, 2.64]

for swel, mu, sig in zip([nom_lo, nom_hi], mus, sigs):
    x = np.linspace(mu - 4*sig, mu + 4*sig, 100)
    y = norm(mu, sig).pdf(x)
    plt.plot(x, y, 'r', label='Obs. with noise')
    plt.fill_between(x, y, color='r', alpha=0.5)

    sns.histplot(swel, binwidth=0.5,
                 ec='k', stat='density', label='Forward propagation')

    plt.xlim([mu - 4*sig, mu + 4*sig])

    plt.xlabel(r'Fuel Swelling (\%)')
    plt.legend(fontsize=10)

    plt.tight_layout()
    plt.show()

# Changes in swelling range

In [ ]:
def swel_change(y_now, y_before, binw=0.5):
    sns.histplot(y_before, binwidth=binw,
                 stat='density', label='before')

    sns.histplot(y_now, binwidth=binw,
                 alpha=0.7, stat='density', label='after')

    plt.legend()
    plt.show()

In [ ]:
swel_change(swel_lo, y_train_lo, 0.2)

In [ ]:
swel_change(swel_hi, y_train_hi, 2)

# Everything together

In [ ]:
# Before IUQ
print(mean(y_train_lo), stdev(y_train_lo))
print(mean(y_train_hi), stdev(y_train_hi))

In [ ]:
# After IUQ
print(mean(swel_lo), stdev(swel_lo))
print(mean(swel_hi), stdev(swel_hi))

In [ ]:
x_labels = [r'Low $F_d$', 'High $F_d$']
x_indices = np.arange(len(x_labels))

# [low F_d, high_Fd]
means = {
    'Observation': [10.7, 32.0],
    'DART': [11.93, 49.41],
    'FUQ': [10.49, 31.99]
}

# [low F_d, high_Fd]
sdevs = {
    'Observation': [2.64, 2.64],
    'DART': [2.71, 19.43],
    'FUQ': [1.47, 2.72]
}

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
width = 0.1

fig, ax = plt.subplots()

for i, (label, mean_vals) in enumerate(means.items()):
    sd_vals = sdevs[label]
    pos = x_indices + (i - 1) * width
    
    ax.errorbar(pos, mean_vals, yerr=sd_vals, fmt='o', label=label, 
                color=colors[i], capsize=6, elinewidth=2, markeredgewidth=2)
    
    for j in range(len(pos)):
        ax.add_patch(plt.Rectangle(
            (pos[j] - width/4, mean_vals[j] - sd_vals[j]),
            width/2, 2 * sd_vals[j],
            fill=True, color=colors[i], alpha=0.1, lw=0
        ))

ax.set_xticks(x_indices)
ax.set_xticklabels(x_labels)
ax.set_ylabel(r'Fuel swelling (\%)')
ax.legend()

plt.tight_layout()
plt.show()
#plt.savefig('aftermath.pdf')